In [15]:
csv_path='data (1) (1).csv'
df = pd.read_csv(csv_path)
df=df.drop(columns=['POS_Tags','POS_Encoded',],axis=1)
df.head()

,text,label,Label_encoded
0,خیلی ناراحت شدم وقتی خبر بدی شنیدم 236.,sadness,5
1,یک فرشته به خواب یکنفر میاد و بهش میگه خدا گفت...,neutral,4
2,ترسیدم چون صدای عجیبی شنیدم 12201.,fear,2
3,اونقدر حواسمون بود که بقیه ناراحت نشن که الان ...,sadness,5
4,خیلی راحته که ... کدوم جالش..,neutral,4


In [16]:
df=df.dropna(subset=['label'])

In [17]:
import pandas as pd
import numpy as np


import pandas as pd
import re

df = pd.DataFrame(df)

# Function to clean text
def clean_text(text):
    # 1. Remove @mentions
    text = re.sub(r'@\w+', '', text)
    # 2. Remove English words (sequences of a-z or A-Z)
    text = re.sub(r'[a-zA-Z]+', '', text)
    # 3. Remove single English letters
    text = re.sub(r'\b[a-zA-Z]\b', '', text)
    # 4. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

import stanza
import pandas as pd


stanza.download("fa")  # run only once
nlp = stanza.Pipeline("fa", processors="tokenize,pos", verbose=False)


# 3. Function: extract raw POS tags
def get_pos_tags(sentence: str):
    doc = nlp(str(sentence))
    return [word.upos for sent in doc.sentences for word in sent.words]

# 4. Build UPOS vocabulary
all_tags = set()
for sentence in df["text"]:   # assumes you have a column named "Sentence"
    all_tags.update(get_pos_tags(sentence))
upos_map = {tag: idx for idx, tag in enumerate(sorted(all_tags))}

# 5. Function: encode POS tags
def get_pos_tags_encoded(sentence: str):
    tags = get_pos_tags(sentence)
    return [upos_map[tag] for tag in tags]

# 6. Add new columns
df["POS_Tags"] = df["text"].apply(get_pos_tags)
df["POS_Encoded"] = df["text"].apply(get_pos_tags_encoded)

# 7. Save back into the same Excel file (overwrite with updated version)
df.to_csv(csv_path, index=False)

# Apply cleaning
df["text"] = df["text"].apply(clean_text)

from sklearn.preprocessing import LabelEncoder
label_encoder=LabelEncoder()
df["Label_encoded"]=label_encoder.fit_transform(df["label"])

df["text"] = df["text"].fillna("").astype(str)

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(analyzer="char",max_features=10000,ngram_range=(3,5))
X_text = tfidf.fit_transform(df["text"])

df["POS_doc"] = df["POS_Tags"].apply(lambda tags: " ".join(tags))


from sklearn.feature_extraction.text import TfidfVectorizer

# simple: unigrams of POS tags
pos_vec = TfidfVectorizer(analyzer="word", vocabulary=sorted(upos_map.keys()))
X_pos = pos_vec.fit_transform(df["POS_doc"])


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:1364: UserWarning: Upper case characters found in vocabulary while 'lowercase' is True. These entries will not be matched with any documents
  warnings.warn(


In [18]:
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split
X = hstack([X_text, X_pos])
y = df["Label_encoded"]

# -----------------------------
# Step 5: Train/Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
clf = LogisticRegression(max_iter=1000, solver="lbfgs", multi_class="multinomial")
clf.fit(X_train, y_train)

# -----------------------------
# Step 7: Evaluation
# -----------------------------
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

       anger       0.52      0.46      0.49      1865
     disgust       0.87      0.40      0.55      1116
        fear       0.80      0.52      0.63      1227
   happiness       0.86      0.46      0.60      1117
     neutral       0.36      0.79      0.50      2270
     sadness       0.63      0.42      0.50      1301
    surprize       0.88      0.50      0.64      1027

    accuracy                           0.53      9923
   macro avg       0.70      0.51      0.56      9923
weighted avg       0.65      0.53      0.54      9923



In [19]:
# Iteration 2 Logistic
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",   # default in new versions
    n_jobs=-1,
    class_weight="balanced",
     multi_class="multinomial" # 🔑 balances minority vs majority classes
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

       anger       0.53      0.46      0.49      1865
     disgust       0.54      0.44      0.49      1116
        fear       0.60      0.57      0.58      1227
   happiness       0.61      0.54      0.57      1117
     neutral       0.39      0.52      0.45      2270
     sadness       0.49      0.47      0.48      1301
    surprize       0.61      0.54      0.57      1027

    accuracy                           0.50      9923
   macro avg       0.54      0.51      0.52      9923
weighted avg       0.52      0.50      0.51      9923



In [20]:
# Iteration 3
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))
print("Custom class weights:", class_weights)

clf = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    n_jobs=-1,
    class_weight=class_weights,
    multi_class="multinomial"
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

Custom class weights: {0: 0.8156131842840704, 1: 1.2617140314069553, 2: 1.1894572807096406, 3: 1.2936670903816694, 4: 0.6102176987885124, 5: 1.008383933335027, 6: 1.3564935064935064}


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

       anger       0.53      0.46      0.49      1865
     disgust       0.54      0.44      0.49      1116
        fear       0.60      0.57      0.58      1227
   happiness       0.61      0.54      0.57      1117
     neutral       0.39      0.52      0.45      2270
     sadness       0.49      0.47      0.48      1301
    surprize       0.61      0.54      0.57      1027

    accuracy                           0.50      9923
   macro avg       0.54      0.51      0.52      9923
weighted avg       0.52      0.50      0.51      9923



In [ ]:
# Iteration 4



In [ ]:
import re
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.pipeline import FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin


def fa_normalize(s: str) -> str:
    if not isinstance(s, str):
        s = "" if s is None else str(s)
    s = (s.replace("\u064A", "\u06CC")   # Arabic Yeh -> Farsi Yeh
           .replace("\u0643", "\u06A9")   # Arabic Kaf -> Keheh
           .replace("\u0640", ""))        # Tatweel
    s = re.sub(r"[\u200C\u200D\u200E\u200F]", " ", s)   # zero-widths -> space
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text"] = df["text"].fillna("").map(fa_normalize)

# -------------------------
# 1) Labels -> integers
# -------------------------
CANDIDATES = ["label", "Label", "emotion", "Emotion", "target", "y"]
label_col = next((c for c in CANDIDATES if c in df.columns), None)
assert label_col is not None, f"Label column not found. Have: {list(df.columns)}"

le = LabelEncoder()
y = le.fit_transform(df[label_col].astype(str))

# -------------------------
# 2) Text feature extractors
# -------------------------
# char n-grams (great for Persian morphology/orthography)
char_tfidf = TfidfVectorizer(analyzer="char", ngram_range=(3,5), max_features=20000, min_df=3)

# word n-grams (captures lexical cues; keep modest size)
word_tfidf = TfidfVectorizer(analyzer="word", ngram_range=(1,2), max_features=30000, min_df=2)

X_char = char_tfidf.fit_transform(df["text"])
X_word = word_tfidf.fit_transform(df["text"])

# -------------------------
# 3) POS TF-IDF (use your existing UPOS tags)
#    You already have df["POS_Tags"] as list[str]
# -------------------------
assert "POS_Tags" in df.columns, "Expected df['POS_Tags'] with UPOS tags"
df["POS_doc"] = df["POS_Tags"].apply(lambda tags: " ".join(tags))
pos_tfidf = TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=2)
X_pos = pos_tfidf.fit_transform(df["POS_doc"])

# Combine: [char-grams | word-grams | POS-ngrams]
from scipy.sparse import csr_matrix
X = hstack([X_char, X_word, X_pos], format="csr")

# -------------------------
# 4) Train/Test split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -------------------------
# 5) Class weights: try balanced + smoothed custom
# -------------------------
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
base_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
# Exponent smoothing: alpha in {0.5, 0.75, 1.0}
def weights_with_alpha(alpha):
    w = np.power(base_weights, alpha)
    w = w * (len(w) / w.sum())  # normalize roughly
    return dict(zip(classes, w))

weight_sets = {
    "none": None,
    "balanced": "balanced",
    "alpha_0.5": weights_with_alpha(0.5),
    "alpha_0.75": weights_with_alpha(0.75),
    "alpha_1.0": weights_with_alpha(1.0),
}

# -------------------------
# 6) Small hyperparam search (macro-F1)
# -------------------------
from sklearn.model_selection import cross_val_score

c_vals = [0.25, 0.5, 1.0, 2.0]
solvers = [("lbfgs","l2"), ("saga","l2"), ("saga","l1")]  # l1 needs saga
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best = {"cv_f1": -1, "clf": None, "params": None}
for cw_name, cw in weight_sets.items():
    for C in c_vals:
        for solver, penalty in solvers:
            if penalty == "l1" and solver != "saga":
                continue
            clf = LogisticRegression(
                C=C,
                max_iter=2000,
                solver=solver,
                penalty=penalty,
                n_jobs=-1,
                class_weight=cw
            )
            scores = cross_val_score(clf, X_train, y_train, scoring="f1_macro", cv=cv, n_jobs=-1)
            m = scores.mean()
            # print(f"[{cw_name}] solver={solver} pen={penalty} C={C} -> macroF1={m:.4f}")
            if m > best["cv_f1"]:
                best = {"cv_f1": m, "clf": clf, "params": (cw_name, solver, penalty, C)}

print("Best CV macro-F1:", round(best["cv_f1"],4), "with params:", best["params"])

# -------------------------
# 7) Fit best and evaluate on hold-out
# -------------------------
best["clf"].fit(X_train, y_train)
y_pred = best["clf"].predict(X_test)

print(classification_report(y_test, y_pred, target_names=le.classes_))

# Confusion matrix (optional: inspect most confused pairs)
cm = confusion_matrix(y_test, y_pred, labels=classes)
print("Confusion matrix shape:", cm.shape)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
